# Классификация персонажей One Piece с помощью CNN
## Трансферное обучение: ResNet50 vs EfficientNetB0

**Задача:** Мультиклассовая классификация изображений персонажей аниме One Piece (18 классов)

## 0. Установка зависимостей

In [1]:
# !pip install tensorflow matplotlib scikit-learn seaborn numpy pandas pillow

## 1. Импорты и конфигурация

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Воспроизводимость
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: Произошел сбой в программе инициализации библиотеки динамической компоновки (DLL).


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## 2. Конфигурация датасета

**⬇ Укажите путь к датасету в переменной `DATASET_PATH`**

In [ ]:
import kagglehub

# Download latest version
DATASET_PATH = kagglehub.dataset_download("ibrahimserouis99/one-piece-image-classifier")

print("Path to dataset files:", DATASET_PATH)

# Список персонажей
CLASS_NAMES = [
    'Ace', 'Akainu', 'Brook', 'Chopper', 'Crocodile',
    'Franky', 'Jinbei', 'Kurohige', 'Law', 'Luffy',
    'Mihawk', 'Nami', 'Rayleigh', 'Robin', 'Sanji',
    'Shanks', 'Usopp', 'Zoro'
]

# Гиперпараметры
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 18

# Этап 1: обучение «головы»
EPOCHS_STAGE1 = 15
LR_STAGE1     = 1e-3

# Этап 2: fine-tuning
EPOCHS_STAGE2   = 15
LR_STAGE2       = 1e-5
UNFREEZE_LAYERS = 30   # сколько верхних слоёв базы размораживать

print(f"Dataset path: {DATASET_PATH}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

## 3. Исследовательский анализ данных (EDA)

In [ ]:
dataset_dir = Path(DATASET_PATH)

# Подсчёт изображений по классам
class_counts = {}
for cls in CLASS_NAMES:
    cls_dir = dataset_dir / cls
    if cls_dir.exists():
        images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg'))
        class_counts[cls] = len(images)
    else:
        class_counts[cls] = 0

total_images = sum(class_counts.values())
print(f"Всего изображений: {total_images}")
print("\nРаспределение по классам:")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<12}: {count:>4} изображений")

In [ ]:
# Визуализация распределения классов
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab20(np.linspace(0, 1, len(CLASS_NAMES)))
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Распределение изображений по классам (персонажам)', fontsize=14, fontweight='bold')
ax.set_xlabel('Персонаж', fontsize=12)
ax.set_ylabel('Количество изображений', fontsize=12)
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Отображение примеров изображений для каждого класса
from PIL import Image

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.flatten()

for idx, cls in enumerate(CLASS_NAMES):
    cls_dir = dataset_dir / cls
    images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg'))
    if images:
        img = Image.open(images[0]).convert('RGB').resize((112, 112))
        axes[idx].imshow(img)
        axes[idx].set_title(cls, fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Примеры изображений по каждому классу', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Подготовка данных и аугментация

In [ ]:
# Аугментация тренировочных данных
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,           # случайный поворот ±20°
    width_shift_range=0.15,      # горизонтальный сдвиг
    height_shift_range=0.15,     # вертикальный сдвиг
    shear_range=0.1,             # сдвиг сдвига
    zoom_range=0.15,             # случайный зум
    horizontal_flip=True,        # горизонтальное отражение
    brightness_range=[0.8, 1.2], # коррекция яркости
    fill_mode='nearest',
    validation_split=0.2          # 80% train / 20% val
)

# Валидационные данные — только нормализация
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Тестовые данные — только нормализация (без split)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

print(f"\nТренировочных батчей : {len(train_generator)}")
print(f"Валидационных батчей : {len(val_generator)}")
print(f"Маппинг классов: {train_generator.class_indices}")

In [ ]:
# Визуализация аугментированных изображений
batch_images, batch_labels = next(train_generator)
class_idx_to_name = {v: k for k, v in train_generator.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_images[i])
    label = class_idx_to_name[np.argmax(batch_labels[i])]
    ax.set_title(label, fontsize=10)
    ax.axis('off')
plt.suptitle('Примеры аугментированных изображений (тренировочный батч)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Построение моделей
### 5.1 Вспомогательные функции

In [ ]:
def build_model(base_model, num_classes=NUM_CLASSES, model_name='model'):
    """
    Строит модель для трансферного обучения.
    Замораживает базовую модель и добавляет классификационную «голову».
    """
    # Заморозка всех слоёв базовой модели
    base_model.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='input')
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(512, activation='relu', name='dense_512')(x)
    x = layers.BatchNormalization(name='bn_512')(x)
    x = layers.Dropout(0.4, name='dropout_1')(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(0.3, name='dropout_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name=model_name)
    return model


def unfreeze_top_layers(model, base_model, n_layers):
    """Размораживает n_layers верхних слоёв базовой модели для fine-tuning."""
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    frozen = sum(1 for l in base_model.layers if not l.trainable)
    trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f"  Заморожено слоёв: {frozen} | Разморожено: {trainable}")


def get_callbacks(model_name):
    """Стандартный набор колбэков."""
    return [
        callbacks.ModelCheckpoint(
            f'{model_name}_best.h5',
            monitor='val_accuracy', save_best_only=True, verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=7,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.3,
            patience=3, min_lr=1e-7, verbose=1
        )
    ]


def plot_training_history(history_stage1, history_stage2, model_name):
    """Отрисовывает графики обучения для обоих этапов."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'История обучения — {model_name}', fontsize=14, fontweight='bold')

    # Объединяем истории
    acc  = history_stage1.history['accuracy']  + history_stage2.history['accuracy']
    val_acc = history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']
    loss = history_stage1.history['loss']      + history_stage2.history['loss']
    val_loss = history_stage1.history['val_loss']    + history_stage2.history['val_loss']
    split_epoch = len(history_stage1.history['accuracy'])
    epochs = range(1, len(acc) + 1)

    for ax, metric, val_metric, title in zip(
        axes,
        [acc, loss], [val_acc, val_loss],
        ['Accuracy', 'Loss']
    ):
        ax.plot(epochs, metric, 'b-o', markersize=3, label=f'Train {title}')
        ax.plot(epochs, val_metric, 'r-o', markersize=3, label=f'Val {title}')
        ax.axvline(split_epoch, color='green', linestyle='--', linewidth=1.5, label='Fine-tuning start')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name}_history.png', dpi=150, bbox_inches='tight')
    plt.show()


def evaluate_model(model, generator, class_names):
    """Вычисляет все метрики на тестовом генераторе."""
    generator.reset()
    y_pred_prob = model.predict(generator, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = generator.classes[:len(y_pred)]

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-score  : {f1:.4f}")
    print("\n", classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    return y_true, y_pred, y_pred_prob, {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    """Отрисовывает confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    for ax, data, title, fmt in zip(
        axes, [cm, cm_norm],
        [f'Confusion Matrix — {model_name} (absolute)', f'Confusion Matrix — {model_name} (normalized)'],
        ['d', '.2f']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.3)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.savefig(f'{model_name}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()


print("Вспомогательные функции определены.")

### 5.2 Модель 1: ResNet50

In [ ]:
print("=" * 60)
print("МОДЕЛЬ 1: ResNet50")
print("=" * 60)

resnet_base = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(*IMG_SIZE, 3)
)

resnet_model = build_model(resnet_base, model_name='ResNet50')
resnet_model.summary()

trainable_params = sum([np.prod(v.shape) for v in resnet_model.trainable_weights])
total_params     = sum([np.prod(v.shape) for v in resnet_model.weights])
print(f"\nТренируемых параметров (Этап 1): {trainable_params:,}")
print(f"Всего параметров               : {total_params:,}")

In [ ]:
# RESNET — Этап 1: обучение только «головы»
resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_STAGE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- ResNet50: Этап 1 — обучение головы ---")
history_resnet_s1 = resnet_model.fit(
    train_generator,
    epochs=EPOCHS_STAGE1,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet50'),
    verbose=1
)

In [ ]:
# RESNET — Этап 2: fine-tuning
print("\n--- ResNet50: Этап 2 — fine-tuning ---")
unfreeze_top_layers(resnet_model, resnet_base, UNFREEZE_LAYERS)

resnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_STAGE2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_resnet_s2 = resnet_model.fit(
    train_generator,
    epochs=EPOCHS_STAGE2,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet50_ft'),
    verbose=1
)

In [ ]:
plot_training_history(history_resnet_s1, history_resnet_s2, 'ResNet50')

### 5.3 Модель 2: EfficientNetB0

In [ ]:
print("=" * 60)
print("МОДЕЛЬ 2: EfficientNetB0")
print("=" * 60)

# EfficientNet имеет встроенную нормализацию, rescale не нужен
train_gen_eff = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
).flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=SEED, shuffle=True
)

val_gen_eff = ImageDataGenerator(
    validation_split=0.2
).flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=SEED, shuffle=False
)

effnet_base = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(*IMG_SIZE, 3)
)

effnet_model = build_model(effnet_base, model_name='EfficientNetB0')
effnet_model.summary()

trainable_params = sum([np.prod(v.shape) for v in effnet_model.trainable_weights])
total_params     = sum([np.prod(v.shape) for v in effnet_model.weights])
print(f"\nТренируемых параметров (Этап 1): {trainable_params:,}")
print(f"Всего параметров               : {total_params:,}")

In [ ]:
# EFFICIENTNET — Этап 1
effnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_STAGE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- EfficientNetB0: Этап 1 — обучение головы ---")
history_eff_s1 = effnet_model.fit(
    train_gen_eff,
    epochs=EPOCHS_STAGE1,
    validation_data=val_gen_eff,
    callbacks=get_callbacks('efficientnet'),
    verbose=1
)

In [ ]:
# EFFICIENTNET — Этап 2: fine-tuning
print("\n--- EfficientNetB0: Этап 2 — fine-tuning ---")
unfreeze_top_layers(effnet_model, effnet_base, UNFREEZE_LAYERS)

effnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_STAGE2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_eff_s2 = effnet_model.fit(
    train_gen_eff,
    epochs=EPOCHS_STAGE2,
    validation_data=val_gen_eff,
    callbacks=get_callbacks('efficientnet_ft'),
    verbose=1
)

In [ ]:
plot_training_history(history_eff_s1, history_eff_s2, 'EfficientNetB0')

## 6. Оценка моделей на валидационной выборке

In [ ]:
print("=" * 50)
print("Оценка ResNet50")
print("=" * 50)
y_true_r, y_pred_r, y_prob_r, metrics_resnet = evaluate_model(resnet_model, val_generator, CLASS_NAMES)

print("\n" + "=" * 50)
print("Оценка EfficientNetB0")
print("=" * 50)
y_true_e, y_pred_e, y_prob_e, metrics_effnet = evaluate_model(effnet_model, val_gen_eff, CLASS_NAMES)

In [ ]:
plot_confusion_matrix(y_true_r, y_pred_r, CLASS_NAMES, 'ResNet50')
plot_confusion_matrix(y_true_e, y_pred_e, CLASS_NAMES, 'EfficientNetB0')

## 7. Сравнение моделей

In [ ]:
# Таблица сравнения
comparison_df = pd.DataFrame({
    'Метрика': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1-score (weighted)'],
    'ResNet50': [
        metrics_resnet['accuracy'], metrics_resnet['precision'],
        metrics_resnet['recall'],   metrics_resnet['f1']
    ],
    'EfficientNetB0': [
        metrics_effnet['accuracy'], metrics_effnet['precision'],
        metrics_effnet['recall'],   metrics_effnet['f1']
    ]
})
comparison_df['Лучшая'] = comparison_df.apply(
    lambda row: 'ResNet50' if row['ResNet50'] > row['EfficientNetB0'] else 'EfficientNetB0', axis=1
)
print(comparison_df.to_string(index=False))
comparison_df.to_csv('model_comparison.csv', index=False)

In [ ]:
# Визуальное сравнение метрик
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-score']
resnet_values = [metrics_resnet['accuracy'], metrics_resnet['precision'],
                 metrics_resnet['recall'],   metrics_resnet['f1']]
effnet_values = [metrics_effnet['accuracy'], metrics_effnet['precision'],
                 metrics_effnet['recall'],   metrics_effnet['f1']]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, resnet_values, width, label='ResNet50',       color='steelblue',   edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, effnet_values, width, label='EfficientNetB0', color='darkorange',  edgecolor='black', linewidth=0.5)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.1)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylabel('Значение метрики', fontsize=12)
ax.set_title('Сравнение ResNet50 и EfficientNetB0', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Вывод победителя
best_model = 'ResNet50' if metrics_resnet['f1'] >= metrics_effnet['f1'] else 'EfficientNetB0'
best_f1    = max(metrics_resnet['f1'], metrics_effnet['f1'])
print(f"\n✅ Лучшая модель по F1-score: {best_model} ({best_f1:.4f})")

In [ ]:
# Сравнение кривых обучения — accuracy на валидации
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (h_s1, h_s2, name, color) in zip(axes, [
    (history_resnet_s1, history_resnet_s2, 'ResNet50',       'steelblue'),
    (history_eff_s1,    history_eff_s2,    'EfficientNetB0', 'darkorange')
]):
    val_acc = h_s1.history['val_accuracy'] + h_s2.history['val_accuracy']
    train_acc = h_s1.history['accuracy']  + h_s2.history['accuracy']
    split = len(h_s1.history['accuracy'])
    epochs = range(1, len(val_acc) + 1)

    ax.plot(epochs, train_acc, linestyle='--', color=color, alpha=0.6, label='Train Accuracy')
    ax.plot(epochs, val_acc, color=color, linewidth=2, label='Val Accuracy')
    ax.axvline(split, color='green', linestyle=':', linewidth=1.5, label='Fine-tuning start')
    ax.set_title(f'{name}: Кривая обучения', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Grad-CAM — визуализация внимания модели

In [ ]:
import cv2

def get_gradcam_heatmap(model, img_array, last_conv_layer_name):
    """Вычисляет Grad-CAM тепловую карту."""
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_class = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_class]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_class.numpy()


def overlay_gradcam(img_path, heatmap, alpha=0.4):
    """Накладывает Grad-CAM тепловую карту на изображение."""
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, IMG_SIZE)
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)
    return cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)


# Последние свёрточные слои для каждой архитектуры
RESNET_LAST_CONV  = 'conv5_block3_out'      # ResNet50
EFFNET_LAST_CONV  = 'top_conv'              # EfficientNetB0

print("Grad-CAM функции определены.")

In [ ]:
from PIL import Image as PILImage

# Собираем по 1 примеру для 6 случайных персонажей
sample_chars = np.random.choice(CLASS_NAMES, 6, replace=False)
sample_images = []
for cls in sample_chars:
    imgs = list((dataset_dir / cls).glob('*.jpg')) + list((dataset_dir / cls).glob('*.png'))
    if imgs:
        sample_images.append((imgs[0], cls))

fig, axes = plt.subplots(len(sample_images), 3, figsize=(15, 5 * len(sample_images)))
if len(sample_images) == 1:
    axes = [axes]

for row, (img_path, true_cls) in enumerate(sample_images):
    orig = np.array(PILImage.open(img_path).convert('RGB').resize(IMG_SIZE))
    img_norm = orig.astype('float32') / 255.0
    img_arr  = np.expand_dims(img_norm, 0)

    try:
        hm_r, pred_r = get_gradcam_heatmap(resnet_model, img_arr, RESNET_LAST_CONV)
        cam_r = overlay_gradcam(img_path, hm_r)
    except Exception as e:
        cam_r = orig
        pred_r = -1
        print(f"ResNet Grad-CAM error: {e}")

    try:
        hm_e, pred_e = get_gradcam_heatmap(effnet_model, img_arr, EFFNET_LAST_CONV)
        cam_e = overlay_gradcam(img_path, hm_e)
    except Exception as e:
        cam_e = orig
        pred_e = -1
        print(f"EfficientNet Grad-CAM error: {e}")

    pred_name_r = CLASS_NAMES[pred_r] if pred_r >= 0 else '?'
    pred_name_e = CLASS_NAMES[pred_e] if pred_e >= 0 else '?'

    axes[row][0].imshow(orig)
    axes[row][0].set_title(f'Original\nTrue: {true_cls}', fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(cam_r)
    color_r = 'green' if pred_name_r == true_cls else 'red'
    axes[row][1].set_title(f'ResNet50 Grad-CAM\nPred: {pred_name_r}', fontsize=10, color=color_r)
    axes[row][1].axis('off')

    axes[row][2].imshow(cam_e)
    color_e = 'green' if pred_name_e == true_cls else 'red'
    axes[row][2].set_title(f'EfficientNetB0 Grad-CAM\nPred: {pred_name_e}', fontsize=10, color=color_e)
    axes[row][2].axis('off')

plt.suptitle('Grad-CAM визуализация: зоны внимания моделей', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Анализ ошибок

In [ ]:
# Находим неверно классифицированные примеры
def get_misclassified(y_true, y_pred, generator, n=8):
    wrong_indices = np.where(y_true != y_pred)[0]
    sample_idx = np.random.choice(wrong_indices, min(n, len(wrong_indices)), replace=False)
    return sample_idx

val_generator.reset()
all_images = []
for i in range(len(val_generator)):
    batch, _ = val_generator[i]
    all_images.extend(batch)
all_images = np.array(all_images[:len(y_true_r)])

wrong_idx_r = get_misclassified(y_true_r, y_pred_r, val_generator)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, idx in enumerate(wrong_idx_r[:8]):
    axes[i].imshow(all_images[idx])
    true_name  = CLASS_NAMES[y_true_r[idx]]
    pred_name  = CLASS_NAMES[y_pred_r[idx]]
    confidence = y_prob_r[idx][y_pred_r[idx]]
    axes[i].set_title(f'True: {true_name}\nPred: {pred_name}\nConf: {confidence:.2f}',
                      fontsize=9, color='red')
    axes[i].axis('off')

plt.suptitle('Примеры ошибочных предсказаний — ResNet50', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('misclassified_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Итоговый отчёт

In [ ]:
print("=" * 60)
print("ИТОГОВЫЙ ОТЧЁТ")
print("=" * 60)
print(f"\nЗадача: классификация персонажей One Piece ({NUM_CLASSES} классов)")
print(f"Всего изображений в датасете: {total_images}")
print(f"Размер изображений: {IMG_SIZE}")
print(f"Размер батча: {BATCH_SIZE}")
print()

print("Метрики ResNet50:")
for k, v in metrics_resnet.items():
    print(f"  {k:<12}: {v:.4f}")

print()
print("Метрики EfficientNetB0:")
for k, v in metrics_effnet.items():
    print(f"  {k:<12}: {v:.4f}")

print()
winner = 'ResNet50' if metrics_resnet['f1'] >= metrics_effnet['f1'] else 'EfficientNetB0'
print(f">>> Победитель по F1-score: {winner}")
delta = abs(metrics_resnet['f1'] - metrics_effnet['f1'])
print(f">>> Разница в F1-score: {delta:.4f}")

In [ ]:
# Сохранение финальных моделей
resnet_model.save('resnet50_onepiece_final.h5')
effnet_model.save('efficientnetb0_onepiece_final.h5')
print("Модели сохранены.")